In [0]:
import pandas as pd
import io
from pyspark.sql.functions import lit, current_timestamp
from pyspark.sql.types import StringType
from azure.storage.blob import BlobServiceClient

In [0]:
AZURE_STORAGE_ACCOUNT = dbutils.secrets.get(scope="my-api-keys", key="AZURE_STORAGE_ACCOUNT")
AZURE_STORAGE_KEY     = dbutils.secrets.get(scope="my-api-keys", key="AZURE_STORAGE_KEY")
AZURE_CONTAINER       = dbutils.secrets.get(scope="my-api-keys", key="AZURE_CONTAINER")

In [0]:
CATALOG = "file_upload"
SCHEMA  = "bronze"
TABLE   = "sales_raw" #utworzy sam przy SaveAsTable

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

In [0]:
connection_string = (
    f"DefaultEndpointsProtocol=https;"
    f"AccountName={AZURE_STORAGE_ACCOUNT};"
    f"AccountKey={AZURE_STORAGE_KEY};"
    f"EndpointSuffix=core.windows.net"
)
blob_service     = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service.get_container_client(AZURE_CONTAINER)


blobs = list(container_client.list_blobs())
files = [b.name for b in blobs if b.name.endswith((".csv", ".xlsx"))]
print(f"Znalezione pliki: {files}")

In [0]:
# ── Przetwarzanie każdego pliku ─────────────────────────────
for filename in files:
    blob_client  = container_client.get_blob_client(filename)
    file_bytes   = blob_client.download_blob().readall()

    if filename.endswith(".csv"):
        text = file_bytes.decode("utf-8-sig", errors="ignore")
        pdf  = pd.read_csv(io.StringIO(text), sep=";", dtype=str, encoding_errors="ignore")
    elif filename.endswith(".xlsx"):
        pdf = pd.read_excel(io.BytesIO(file_bytes), dtype=str)

    pdf.columns = pdf.columns.str.strip()
    pdf = pdf.loc[:, ~pdf.columns.str.startswith('Unnamed')]

    df = spark.createDataFrame(pdf)
    df = df \
        .withColumn("source_file", lit(filename)) \
        .withColumn("loaded_at", current_timestamp())

    for col_name, dtype in df.dtypes:
        if dtype in ('void', 'null'):
            df = df.withColumn(col_name, df[col_name].cast(StringType()))

    df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE}")

    print(f"Zaladowano {df.count()} wierszy z {filename}")

print("Bronze complete.")

Usuwanie pliku z repo

In [0]:
# ── Usuń przetworzone pliki z Blob Storage ──────────────────
for filename in files:
    container_client.delete_blob(filename)
    print(f"Usunieto: {filename}")